# Homework 2 — European Football Archive

**Goal:** Scrape the assigned local website with Beautiful Soup and save seven connected CSV files.

This notebook is set up for **Option B: Google Colab**, following the website setup guide. It uploads the assigned website ZIP, extracts it inside the Colab runtime, reads the saved HTML files directly, and writes the CSV outputs.

**Before running:** Have the complete assigned website ZIP ready to upload. Run the cells from top to bottom. The ZIP should contain the homepage (`index.html`) and all the website's subfolders.

In [ ]:
# Option B, Step 1: Install packages and upload the website ZIP
%pip -q install beautifulsoup4 requests

from google.colab import files
uploaded = files.upload()  # Choose your assigned website ZIP

Saving website.zip to website (1).zip


In [20]:
# Option B, Step 2: Extract the ZIP inside the Colab runtime
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO

zip_name = next(iter(uploaded))
destination = Path("/content/site")

with ZipFile(BytesIO(uploaded[zip_name])) as archive:
    archive.extractall(destination)

# Locate index.html, including when the ZIP contains an outer folder.
entries = list(destination.rglob("index.html"))
if not entries:
    raise FileNotFoundError(
        "No index.html was found in the uploaded ZIP. Upload the complete assigned website ZIP."
    )

entry = min(entries, key=lambda path: len(path.parts))
website = entry.parent

print("Homepage:", entry)
print("Website folder:", website)
del uploaded  # Release the uploaded ZIP bytes from memory

NameError: name 'uploaded' is not defined

In [ ]:
# Option B, Step 3: Read and parse the homepage
import re
import pandas as pd
from bs4 import BeautifulSoup

html = entry.read_text(encoding="utf-8")
soup = BeautifulSoup(html, "html.parser")

print(soup.title.get_text(" ", strip=True) if soup.title else "(no title)")
print("Links on this page:", len(soup.select("a[href]")))

Welcome · TOUCHLINE
Links on this page: 20


## 1. Explore the website



In [ ]:
for link in soup.select("nav.topnav a"):
    print(link.get_text(" ", strip=True), "->", link.get("href"))

for folder in ["country", "league", "match", "player", "player-attributes", "team", "team-attributes"]:
    folder_path = website / folder
    print(folder, "folder found:", folder_path.is_dir(),
          "| HTML listing pages:", len(list(folder_path.glob("*.html"))) if folder_path.is_dir() else 0)

Countries -> country/index.html
Competitions -> league/index.html
Match centre -> match/index.html
Players -> player/index.html
Player ratings -> player-attributes/index.html
Clubs -> team/index.html
Tactical profiles -> team-attributes/index.html
country folder found: True | HTML listing pages: 1
league folder found: True | HTML listing pages: 1
match folder found: True | HTML listing pages: 1085
player folder found: True | HTML listing pages: 308
player-attributes folder found: True | HTML listing pages: 5115
team folder found: True | HTML listing pages: 9
team-attributes folder found: True | HTML listing pages: 27


## 2. Test one record and one table


In [ ]:
sample = BeautifulSoup((website / "player/records/1.html").read_text(encoding="utf-8"), "html.parser")
for item in sample.select("dl dt"):
    print(item.get_text(" ", strip=True), ":", item.find_next_sibling("dd").get_text(" ", strip=True))

Id : 1
Player api id : 505942
Player name : Aaron Appindangoye
Player fifa api id : 218353
Birthday : 1992-02-29 00:00:00
Height · cm : 182.88
Weight · lb : 187


In [ ]:
sample = BeautifulSoup((website / "match/index.html").read_text(encoding="utf-8"), "html.parser")
table = sample.select_one("table")
print("Columns:", [x.get_text(" ", strip=True) for x in table.select("th")])
print("First row:", [x.get_text(" ", strip=True) for x in table.select("tbody tr")[0].select("td")])

Columns: ['Id', 'Country id', 'League id', 'Season', 'Stage', 'Date', 'Match api id', 'Home team api id', 'Away team api id', 'Home team goal', 'Away team goal']
First row: ['1', '1', '1', '2008/2009', '1', '2008-08-17 00:00:00', '492473', '9987', '9993', '1', '1']


## 3. Two small extraction functions


In [ ]:
def clean_name(name):
    name = name.lower().split("·")[0].strip()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

def clean_value(value):
    value = value.strip()
    if value in ["—", "–", "", "None"]:
        return ""
    return value

def read_record(path):
    soup = BeautifulSoup(path.read_text(encoding="utf-8"), "html.parser")
    row = {}
    for dt in soup.select("dl dt"):
        dd = dt.find_next_sibling("dd")
        if dd is not None:
            row[clean_name(dt.get_text(" ", strip=True))] = clean_value(dd.get_text(" ", strip=True))
    return row

def read_table(path):
    soup = BeautifulSoup(path.read_text(encoding="utf-8"), "html.parser")
    table = soup.select_one("table")
    if table is None:
        return []
    names = [clean_name(x.get_text(" ", strip=True)) for x in table.select("thead th")]
    rows = []
    for tr in table.select("tbody tr"):
        values = [clean_value(x.get_text(" ", strip=True)) for x in tr.select("td")]
        if len(values) == len(names):
            rows.append(dict(zip(names, values)))
    return rows

print("Test record:", read_record(website / "country/records/1.html"))
print("Test table:", read_table(website / "match/index.html")[0])

Test record: {'id': '1', 'name': 'Belgium'}
Test table: {'id': '1', 'country_id': '1', 'league_id': '1', 'season': '2008/2009', 'stage': '1', 'date': '2008-08-17 00:00:00', 'match_api_id': '492473', 'home_team_api_id': '9987', 'away_team_api_id': '9993', 'home_team_goal': '1', 'away_team_goal': '1'}


## 4. Countries, leagues, players, and clubs
For these four groups, each record has its own HTML file. We read every record once, including later records and sort by ID. We do not drop identical rows.

In [ ]:
def collect_records(folder):
    rows = []
    for path in sorted((website / folder / "records").glob("*.html")):
        row = read_record(path)
        if "id" in row:
            rows.append(row)
    data = pd.DataFrame(rows)
    return data.sort_values("id", key=lambda x: pd.to_numeric(x)).reset_index(drop=True)

country = collect_records("country")
league = collect_records("league")
player = collect_records("player")
team = collect_records("team")
for name, data in [("Country", country), ("League", league), ("Player", player), ("Team", team)]:
    print(name, data.shape)
print(player.head(3).to_string(index=False))

Country (11, 2)
League (11, 3)
Player (11060, 7)
Team (299, 5)
id player_api_id        player_name player_fifa_api_id            birthday height weight
 1        505942 Aaron Appindangoye             218353 1992-02-29 00:00:00 182.88    187
 2        155782    Aaron Cresswell             189615 1989-12-15 00:00:00 170.18    146
 3        162549        Aaron Doran             186170 1991-05-13 00:00:00 170.18    163


## 5. Tables with several sections
The match table has five tabs, player attributes have five tabs and team attributes have three tabs. Each tab repeats the same record ID. We join the columns by that ID.



In [ ]:
def collect_sections(folder):
    combined = {}
    paths = sorted((website / folder).glob("*.html"))
    for path in paths:
        for row in read_table(path):
            record_id = row.get("id")
            if not record_id:
                continue
            if record_id not in combined:
                combined[record_id] = {"id": record_id}
            for key, value in row.items():
                if key not in combined[record_id] or combined[record_id][key] == "":
                    combined[record_id][key] = value
    data = pd.DataFrame(list(combined.values()))
    return data.sort_values("id", key=lambda x: pd.to_numeric(x)).reset_index(drop=True)

print("Ready to join table sections by record ID")

Ready to join table sections by record ID


In [ ]:
match = collect_sections("match")
print("Match:", match.shape)
print(match[["id", "season", "home_team_goal", "away_team_goal"]].head(3).to_string(index=False))

Match: (25979, 115)
id    season home_team_goal away_team_goal
 1 2008/2009              1              1
 2 2008/2009              0              0
 3 2008/2009              0              3


In [ ]:
player_attributes = collect_sections("player-attributes")
print("Player attributes:", player_attributes.shape)
print(player_attributes[["id", "player_api_id", "date", "overall_rating"]].head(3).to_string(index=False))

Player attributes: (183978, 42)
id player_api_id                date overall_rating
 1        505942 2016-02-18 00:00:00             67
 2        505942 2015-11-19 00:00:00             67
 3        505942 2015-09-21 00:00:00             62


In [13]:
team_attributes = collect_sections("team-attributes")
print("Team attributes:", team_attributes.shape)
print(team_attributes.head(3).to_string(index=False))

Team attributes: (1458, 25)
id chance_creation_passing chance_creation_passing_class chance_creation_crossing chance_creation_crossing_class chance_creation_shooting chance_creation_shooting_class chance_creation_positioning_class defence_pressure defence_pressure_class defence_aggression defence_aggression_class defence_team_width defence_team_width_class defence_defender_line_class team_fifa_api_id team_api_id                date build_up_play_speed build_up_play_speed_class build_up_play_dribbling build_up_play_dribbling_class build_up_play_passing build_up_play_passing_class build_up_play_positioning_class
 1                      60                        Normal                       65                         Normal                       55                         Normal                         Organised               50                 Medium                 55                    Press                 45                   Normal                       Cover              434       

## 6. Compare with the assignment


In [14]:
datasets = {"Country": country, "League": league, "Match": match, "Player": player, "Player_Attributes": player_attributes, "Team": team, "Team_Attributes": team_attributes}
expected = {"Country": (11, 2), "League": (11, 3), "Match": (25979, 115), "Player": (11060, 7), "Player_Attributes": (183978, 42), "Team": (299, 5), "Team_Attributes": (1458, 25)}
for name, data in datasets.items():
    print(name, "actual:", data.shape, "expected:", expected[name], "match:", data.shape == expected[name])

Country actual: (11, 2) expected: (11, 2) match: True
League actual: (11, 3) expected: (11, 3) match: True
Match actual: (25979, 115) expected: (25979, 115) match: True
Player actual: (11060, 7) expected: (11060, 7) match: True
Player_Attributes actual: (183978, 42) expected: (183978, 42) match: True
Team actual: (299, 5) expected: (299, 5) match: True
Team_Attributes actual: (1458, 25) expected: (1458, 25) match: True


## 7. Check later records and relationships
These checks verify that IDs are unique, later pages were included and related records can be connected. API IDs are different from ordinary  IDs. A missing FIFA ID does not mean the record should be discarded, instead keep it empty and use the player/team API ID when possible. Some references may not match a row in the other table, so we report those rather than inventing data.

In [15]:
for name, data in datasets.items():
    print(name, "duplicate record IDs:", data["id"].duplicated().sum(), "last ID:", data["id"].iloc[-1])
print("Last match:", match[["id", "season", "date", "home_team_goal", "away_team_goal"]].tail(1).to_string(index=False))
print("Last player:", player.tail(1).to_string(index=False))

Country duplicate record IDs: 0 last ID: 24558
League duplicate record IDs: 0 last ID: 24558
Match duplicate record IDs: 0 last ID: 25979
Player duplicate record IDs: 0 last ID: 11075
Player_Attributes duplicate record IDs: 0 last ID: 183978
Team duplicate record IDs: 0 last ID: 51606
Team_Attributes duplicate record IDs: 0 last ID: 1458
Last match:    id    season                date home_team_goal away_team_goal
25979 2015/2016 2015-09-23 00:00:00              4              3
Last player:    id player_api_id        player_name player_fifa_api_id            birthday height weight
11075         39902 Zvjezdan Misimovic             102359 1982-06-05 00:00:00 180.34    176


In [16]:
print("League country references missing:", (~league["country_id"].isin(country["id"])).sum())
print("Match league references missing:", (~match["league_id"].isin(league["id"])).sum())
team_ids = set(team["team_api_id"].dropna())
player_ids = set(player["player_api_id"].dropna())
print("Missing match home team references:", (~match["home_team_api_id"].isin(team_ids)).sum())
print("Missing player attribute references:", (~player_attributes["player_api_id"].isin(player_ids)).sum())
print("Missing team attribute references:", (~team_attributes["team_api_id"].isin(team_ids)).sum())
print("Missing player FIFA IDs:", (player["player_fifa_api_id"] == "").sum())

League country references missing: 0
Match league references missing: 0
Missing match home team references: 0
Missing player attribute references: 0
Missing team attribute references: 0
Missing player FIFA IDs: 0


## 8. Save CSV files
One CSV per reference group makes the relationships easy to understand. Running this cell again replaces the same files instead of appending duplicate rows.

In [17]:
output = Path("homework2")
output.mkdir(exist_ok=True)
for name, data in datasets.items():
    filename = output / (name.lower() + ".csv")
    data.to_csv(filename, index=False)
    print(filename, "saved:", len(data), "rows")

homework2/country.csv saved: 11 rows
homework2/league.csv saved: 11 rows
homework2/match.csv saved: 25979 rows
homework2/player.csv saved: 11060 rows
homework2/player_attributes.csv saved: 183978 rows
homework2/team.csv saved: 299 rows
homework2/team_attributes.csv saved: 1458 rows


## 9. Check saved files


In [18]:
for name, data in datasets.items():
    filename = output / (name.lower() + ".csv")
    saved = pd.read_csv(filename, low_memory=False)
    print(filename.name, "rows:", len(saved), "same count:", len(saved) == len(data))

country.csv rows: 11 same count: True
league.csv rows: 11 same count: True
match.csv rows: 25979 same count: True
player.csv rows: 11060 same count: True
player_attributes.csv rows: 183978 same count: True
team.csv rows: 299 same count: True
team_attributes.csv rows: 1458 same count: True


## 10. Download the CSV outputs from Colab



In [19]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

csv_zip = Path("/content/homework2_csv_files.zip")
with ZipFile(csv_zip, "w", ZIP_DEFLATED) as archive:
    for filename in sorted(output.glob("*.csv")):
        archive.write(filename, arcname=filename.name)

print("Created:", csv_zip)
files.download(str(csv_zip))

Created: /content/homework2_csv_files.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>